# Phase 2: GPU Naive Implementation
**CSC14120 - Parallel Programming**

---

## 2.1 Mục tiêu Phase 2

Chuyển đổi (port) toàn bộ các layer từ CPU sang GPU với cài đặt **naive** (cơ bản):
- Mỗi thread xử lý một phần tử đầu ra
- Sử dụng Global Memory cho toàn bộ tính toán
- Chưa áp dụng các kỹ thuật tối ưu (shared memory, tiling, etc.)

## 2.2 Các CUDA Kernel được cài đặt

| Kernel | Mô tả | Thread mapping |
|:-------|:------|:---------------|
| `conv2d_forward_kernel` | Convolution 2D forward | 1 thread = 1 output pixel |
| `conv2d_backward_data_kernel` | Gradient qua input | 1 thread = 1 input pixel |
| `conv2d_backward_weights_kernel` | Gradient qua weights | 1 thread = 1 weight |
| `relu_forward_kernel` | ReLU activation | 1 thread = 1 element |
| `relu_backward_kernel` | ReLU gradient | 1 thread = 1 element |
| `maxpool2d_forward_kernel` | Max Pooling 2x2 | 1 thread = 1 output pixel |
| `maxpool2d_backward_kernel` | MaxPool gradient | 1 thread = 1 output pixel |
| `upsample2d_forward_kernel` | Nearest neighbor upsampling | 1 thread = 1 output pixel |
| `mse_loss_kernel` | MSE Loss với parallel reduction | Warp shuffle reduction |
| `sgd_update_kernel` | SGD weight update | 1 thread = 1 parameter |

## 2.3 Quản lý bộ nhớ GPU

- `cudaMalloc` / `cudaFree` cho device memory
- `cudaMallocHost` cho pinned host memory (faster transfers)
- `cudaMemcpy` với `cudaMemcpyHostToDevice` và `cudaMemcpyDeviceToHost`
- **He Initialization** cho weights: `std = sqrt(2.0 / fan_in)`

---

## Hướng dẫn chạy:
1. Zip thư mục project (không bao gồm data/)
2. Upload file zip lên Colab
3. Chạy tất cả cells

In [ ]:
# Kiểm tra GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Upload và giải nén project
from google.colab import files
import zipfile, os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

project_root = None
for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        project_root = root
        break

if project_root is None:
    raise RuntimeError("Could not find project root containing 'src' directory")

os.chdir(project_root)
print("Current working directory:", os.getcwd())
!ls

In [ ]:
# Download CIFAR-10
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Done!')
!ls data/*.bin

## 2.4 Build Phase 2 (GPU Naive)

**Compiler flags:**
- `-O2`: Optimization level 2
- `-std=c++17`: C++17 standard
- `-arch=sm_75`: Target Tesla T4 GPU (Colab)
- `-lcurand`: Link cuRAND library cho He Initialization

**Lưu ý quan trọng:** Phải link `-lcurand` để He Initialization hoạt động đúng!

In [ ]:
# Build Phase 2 (GPU Naive) - với curand cho He Initialization
!nvcc -O2 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr -Iinclude \
    -lcurand \
    -o gpu_train src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/dataset.cpp
print('Build complete!')

## 2.5 Training

**Hyperparameters:**
- Epochs: 20
- Batch size: 64
- Learning rate: 0.001
- Optimizer: Plain SGD (no momentum, no weight decay)

**Expected performance (Phase 2 Naive):**
- Time per epoch: ~2-5 minutes (tùy GPU)
- Total training: ~40-100 minutes
- Final loss: ~0.01-0.02

In [ ]:
# Train
!./gpu_train --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase2.csv --log-txt phase2.txt --save-weights phase2.weights

## 2.6 Kết quả và Visualization

In [ ]:
# Visualize training results
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase2.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ep['epoch'], ep['loss'], 'b-o')
ax1.set_title('Training Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'g-o')
ax2.set_title('Time per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Time (seconds)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('phase2_results.png', dpi=150)
plt.show()

print("="*50)
print("PHASE 2 NAIVE GPU RESULTS")
print("="*50)
print(f"Best Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep['loss'].iloc[-1]:.6f}")
print(f"Avg Time per Epoch: {ep['epoch_time_sec'].mean():.2f}s")
print(f"Total Training Time: {ep['epoch_time_sec'].sum():.2f}s ({ep['epoch_time_sec'].sum()/60:.2f} min)")
print("="*50)

## 2.7 Kiểm tra Weights đã được khởi tạo đúng

In [ ]:
# Verify weights are properly initialized (not all zeros)
import struct
import numpy as np

path = "phase2.weights"

with open(path, "rb") as f:
    header = f.read(12)
    magic, version, num_layers = struct.unpack("III", header)
    print(f"MAGIC = {hex(magic)}, version = {version}, num_layers = {num_layers}")
    print()
    
    all_good = True
    for li in range(num_layers):
        in_c, out_c, k = struct.unpack("iii", f.read(12))
        
        (w_size,) = struct.unpack("i", f.read(4))
        w_bytes = f.read(4 * w_size)
        w = np.frombuffer(w_bytes, dtype=np.float32)
        
        (b_size,) = struct.unpack("i", f.read(4))
        b_bytes = f.read(4 * b_size)
        b = np.frombuffer(b_bytes, dtype=np.float32)
        
        # Check if weights are not all zeros
        w_nonzero = np.count_nonzero(w)
        w_mean = np.mean(w)
        w_std = np.std(w)
        
        status = "✓ OK" if w_nonzero > 0 else "✗ ALL ZEROS!"
        if w_nonzero == 0:
            all_good = False
        
        print(f"Layer {li}: in_c={in_c}, out_c={out_c}, k={k}")
        print(f"  Weights: {w_size} params, nonzero={w_nonzero}, mean={w_mean:.6f}, std={w_std:.6f} {status}")
        print(f"  Bias: {b_size} params")
        print()

if all_good:
    print("✓ All weights properly initialized with He Initialization!")
else:
    print("✗ WARNING: Some weights are all zeros - He Initialization may have failed!")

In [ ]:
# Download results
files.download('phase2.csv')
files.download('phase2.txt')
files.download('phase2.weights')
files.download('phase2_results.png')

## 2.8 Phân tích Performance Phase 2

### Bottlenecks của Naive Implementation:

1. **Global Memory Bandwidth**: Mỗi thread đọc weights và input từ global memory → latency cao
2. **Không có data reuse**: Cùng một input pixel được đọc nhiều lần bởi các threads khác nhau
3. **Memory access không coalesced hoàn toàn**: Pattern truy cập bộ nhớ chưa tối ưu

### Cải tiến cho Phase 3:

- Shared Memory Tiling cho Convolution
- cuDNN cho optimized convolution algorithms
- Kernel Fusion (Conv + ReLU)
- Vectorized memory access (float4)